In [1]:
import itertools

import sympy as sy
import numpy as np

Tamanho dos vetores

In [2]:
nd = 3
ng = 2

Vetor de exemplo

In [3]:
vd = list(range(1, nd+1))
vg = list(range(1, ng+1))

Grau do polinômio

In [4]:
pd = nd - 1
pg = ng - 1

In [5]:
x = sy.symbols("x")

In [6]:
# sy.factor(x**4-1)

In [7]:
mk0 = [x + i*r for r in range(1, pd + pg - 1) for i in [-1, 1]]
# mk0 = [x + i*2**r for r in range(0, (nd -1) + (ng - 1) - 1) for i in [-1, 1]]
mk0

[x - 1, x + 1]

In [8]:
mk = ([x] + mk0)[:pd + pg]
mk

[x, x - 1, x + 1]

In [9]:
di = sy.symbols(" ".join(f"d{i}"for i in range(nd)))
di

(d0, d1, d2)

In [10]:
gi = sy.symbols(" ".join(f"g{i}"for i in range(ng)))
gi

(g0, g1)

In [11]:
dx = sum([i*x**e for e, i in enumerate(di)])
dx

d0 + d1*x + d2*x**2

In [12]:
gx = sum([i*x**e for e, i in enumerate(gi)])
gx

g0 + g1*x

In [13]:
sx = gx*dx
sx

(g0 + g1*x)*(d0 + d1*x + d2*x**2)

In [14]:
gk_ = [sy.div(gx, q, domain ='QQ')[1] for q in mk]
gk = gk_ + [gx.args[-1].args[0]]
gk

[g0, g0 + g1, g0 - g1, g1]

In [15]:
dk_ = [sy.div(dx, q, domain ='QQ')[1] for q in mk]
dk = dk_ + [dx.args[-1].args[0]]
dk

[d0, d0 + d1 + d2, d0 - d1 + d2, d2]

In [16]:
la = [[d.coeff(c, 1) for c in di] for d in dk]
la

[[1, 0, 0], [1, 1, 1], [1, -1, 1], [0, 0, 1]]

In [17]:
ma = sy.Matrix(la)
ma

Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])

In [18]:
mmk_ = [sy.expand(np.prod(d)) for d in itertools.combinations(reversed(mk), len(mk)-1)]
mmk = mmk_ + [sy.expand(np.prod(mk))]
mmk

[x**2 - 1, x**2 + x, x**2 - x, x**3 - x]

Pegando quociente e resto, agora tem q colocar no formato nm+NM=1

In [19]:
mx_div = [sy.div(dv, ds, domain ='QQ') for dv, ds in zip(mmk, mk)]
mx_div

[(x, -1), (x + 2, 2), (x - 2, 2)]

Multiplicar o resto pela matriz G depois
o sinal negativo vai pra matriz G e não pra C

In [20]:
nnk = [1/z[1] for z in mx_div]
nnk

[-1, 1/2, 1/2]

In [21]:
nk = [q[0]*r*(-1) for q, r in zip(mx_div, nnk)]
nk

[x, -x/2 - 1, 1 - x/2]

colocar em forma matricial

In [22]:
lc = [[d.coeff(x, c) for d in mmk] for c in range(len(np.prod(mk).as_poly().all_coeffs()))]
lc

[[-1, 0, 0, 0], [0, 1, -1, -1], [1, 1, 1, 0], [0, 0, 0, 1]]

In [23]:
cc = sy.Matrix(lc)
cc

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])

In [24]:
lbg = [g*r for g, r in zip(gk, nnk+[1])]
lbg

[-g0, g0/2 + g1/2, g0/2 - g1/2, g1]

In [25]:
bbg = sy.diag(*lbg)
bbg

Matrix([
[-g0,           0,           0,  0],
[  0, g0/2 + g1/2,           0,  0],
[  0,           0, g0/2 - g1/2,  0],
[  0,           0,           0, g1]])

In [26]:
s = sy.MatMul(cc, bbg, ma, sy.Matrix(di))
s

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])*Matrix([
[-g0,           0,           0,  0],
[  0, g0/2 + g1/2,           0,  0],
[  0,           0, g0/2 - g1/2,  0],
[  0,           0,           0, g1]])*Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])*Matrix([
[d0],
[d1],
[d2]])

In [27]:
subs = {k: v for k, v in zip(di+gi, vd + vg)}
subs

{d0: 1, d1: 2, d2: 3, g0: 1, g1: 2}

In [28]:
si = s.subs(subs)
si

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])*Matrix([
[-1,   0,    0, 0],
[ 0, 3/2,    0, 0],
[ 0,   0, -1/2, 0],
[ 0,   0,    0, 2]])*Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])*Matrix([
[1],
[2],
[3]])

In [29]:
sy.expand(sx)

d0*g0 + d0*g1*x + d1*g0*x + d1*g1*x**2 + d2*g0*x**2 + d2*g1*x**3

In [30]:
se = sy.MatMul(cc, bbg, ma, sy.Matrix(di), evaluate=True)
se

Matrix([
[        d0*g0],
[d0*g1 + d1*g0],
[d1*g1 + d2*g0],
[        d2*g1]])

In [31]:
out = np.convolve(vd, vg)
print(out)

[1 4 7 6]


In [32]:
se.subs(subs)

Matrix([
[1],
[4],
[7],
[6]])